# VTM parcial no Taskonomy

Notebook fino sobre o pacote `vtm`. Ative uma GPU em **Runtime → Change runtime type** antes do treino.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

A célula seguinte clona este repositório do GitHub na primeira execução e executa `git pull` nas sessões seguintes.

In [ ]:
from pathlib import Path
import subprocess

repo = Path('/content/Visual-Token-Matching')
if not (repo / '.git').exists():
    subprocess.run([
        'git', 'clone',
        'https://github.com/NataLira1/Visual-Token-Matching.git',
        str(repo),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)

%cd /content/Visual-Token-Matching
!pip install -q -e ".[experiment]"
import torch, timm
print('torch', torch.__version__, 'timm', timm.__version__, 'gpu', torch.cuda.get_device_name(0))

## Download seletivo

O dry-run deve ser inspecionado antes de remover `--dryrun`. O download e o pré-processamento não fazem parte das quatro horas de GPU.

In [ ]:
!sudo apt-get -qq update && sudo apt-get -qq install -y aria2
import subprocess

download_name = input('Nome completo para aceitar os termos do dataset: ').strip()
download_email = input('E-mail válido: ').strip()
download_command = [
    'omnitools.download', 'rgb', 'segment_semantic',
    '--components', 'taskonomy',
    '--subset', 'tiny',
    '--dest', '/content/drive/MyDrive/taskonomy',
    '--connections_total', '16',
    '--agree_all',
    '--name', download_name,
    '--email', download_email,
]
subprocess.run(download_command + ['--dryrun'], check=True)

In [ ]:
# Execute apenas depois de conferir o dry-run:
subprocess.run(download_command, check=True)

## Preparação, meta-treino e avaliação

In [ ]:
from pathlib import Path
import yaml
config = yaml.safe_load(Path('configs/taskonomy_vtm.yaml').read_text())
config['data']['root'] = '/content/drive/MyDrive/taskonomy'
config['data']['manifest'] = '/content/drive/MyDrive/vtm_outputs/taskonomy_manifest.json'
config['train']['checkpoint'] = '/content/drive/MyDrive/vtm_outputs/vtm_best.pt'
config['experiment']['output_dir'] = '/content/drive/MyDrive/vtm_outputs/evaluation'
runtime_config = Path('/content/vtm_taskonomy_runtime.yaml')
runtime_config.write_text(yaml.safe_dump(config, sort_keys=False))
runtime_config

In [ ]:
!vtm-taskonomy prepare --config /content/vtm_taskonomy_runtime.yaml
!vtm-taskonomy train --config /content/vtm_taskonomy_runtime.yaml
!vtm-taskonomy evaluate --config /content/vtm_taskonomy_runtime.yaml

In [ ]:
import pandas as pd
from IPython.display import display, Image
out = Path(config['experiment']['output_dir'])
display(pd.read_csv(out / 'summary.csv'))
display(pd.read_json(out / 'hypothesis.json'))
panels = sorted((out / 'panels').glob('*.png'))
if panels:
    display(Image(filename=str(panels[0])))